# MNIST DC-GAN — 최신 PyTorch 코드

이 노트북은 MNIST 손글씨 숫자 이미지를 학습하여 새로운 숫자 이미지를 생성하는 DC-GAN 예제입니다.

구성은 다음과 같습니다.

1. 라이브러리 불러오기
2. 하이퍼파라미터 설정
3. MNIST 데이터 준비
4. 생성자 모델 설계
5. 감별자 모델 설계
6. 모델 설계 상세 설명
7. 손실함수와 최적화 알고리즘 상세 설명
8. GAN 학습 함수 작성
9. 생성 이미지 저장 및 확인
10. 모델 저장과 불러오기


In [ ]:
# PyTorch의 핵심 패키지입니다.
# 텐서 연산, GPU 연산, 딥러닝 모델 학습에 사용합니다.
import torch

# torch.nn은 신경망 계층, 활성화 함수, 손실함수 등을 제공합니다.
import torch.nn as nn

# torch.optim은 Adam, AdamW, SGD 같은 최적화 알고리즘을 제공합니다.
import torch.optim as optim

# DataLoader는 데이터를 미니배치 단위로 나누어 모델에 공급합니다.
from torch.utils.data import DataLoader

# torchvision.datasets는 MNIST 같은 이미지 데이터셋을 제공합니다.
from torchvision.datasets import MNIST

# torchvision.transforms는 이미지 전처리 기능을 제공합니다.
import torchvision.transforms as transforms

# torchvision.utils는 이미지 그리드 저장과 시각화에 사용합니다.
import torchvision.utils as vutils

# matplotlib은 생성된 이미지를 화면에 출력하는 데 사용합니다.
import matplotlib.pyplot as plt

# os는 폴더 생성과 파일 경로 처리에 사용합니다.
import os

# glob은 특정 패턴에 맞는 파일을 찾는 데 사용합니다.
import glob

# time은 학습 시간을 측정하는 데 사용합니다.
from time import time

# PyTorch 버전을 출력합니다.
print("PyTorch version:", torch.__version__)

# CUDA 사용 가능 여부를 출력합니다.
print("CUDA available:", torch.cuda.is_available())

## 1. 하이퍼파라미터 설정

GAN은 생성자와 감별자가 서로 경쟁하면서 학습하는 구조입니다.

- 생성자: 무작위 노이즈에서 가짜 이미지를 생성합니다.
- 감별자: 입력 이미지가 진짜인지 가짜인지 판별합니다.

하이퍼파라미터는 학습 전에 사람이 직접 정하는 값입니다.


In [ ]:
# 실험 결과를 최대한 재현 가능하게 만들기 위해 난수 시드를 고정합니다.
SEED = 1234

# CPU 연산의 난수 시드를 고정합니다.
torch.manual_seed(SEED)

# GPU가 있는 경우 GPU 난수 시드도 고정합니다.
torch.cuda.manual_seed_all(SEED)

# 전체 학습 반복 횟수입니다.
# 실습에서는 5 정도로 시작하고, 더 좋은 이미지를 얻으려면 20 이상으로 늘릴 수 있습니다.
EPOCHS = 5

# 한 번에 학습할 이미지 개수입니다.
# 너무 크면 메모리를 많이 사용하고, 너무 작으면 학습이 불안정할 수 있습니다.
BATCH_SIZE = 128

# 생성자에 입력할 노이즈 벡터의 차원입니다.
# 생성자는 이 노이즈를 바탕으로 이미지를 만듭니다.
NOISE_DIM = 100

# MNIST 이미지는 흑백 이미지이므로 채널 수는 1입니다.
IMAGE_CHANNELS = 1

# 생성할 이미지 크기입니다.
# MNIST는 28x28 이미지입니다.
IMAGE_SIZE = 28

# Adam 최적화 알고리즘의 학습률입니다.
LEARNING_RATE = 0.0002

# Adam 최적화 알고리즘의 beta1 값입니다.
# DC-GAN에서는 0.5를 자주 사용합니다.
BETA1 = 0.5

# 학습 결과 이미지를 저장할 폴더입니다.
OUTPUT_DIR = "dcgan_output"

# 출력 폴더를 생성합니다.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 기존 출력 이미지를 삭제하여 새 학습 결과만 남깁니다.
for file_path in glob.glob(os.path.join(OUTPUT_DIR, "*.png")):
    # 기존 PNG 파일을 삭제합니다.
    os.remove(file_path)

# GPU가 가능하면 GPU를 사용하고, 그렇지 않으면 CPU를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 사용할 장치를 출력합니다.
print("사용 장치:", device)

## 2. MNIST 데이터 준비

MNIST 이미지는 0부터 9까지의 손글씨 숫자 이미지입니다.  
GAN에서는 정답 라벨을 사용하지 않고, 진짜 이미지 자체만 사용합니다.

이미지는 `[0, 1]` 범위에서 `[-1, 1]` 범위로 정규화합니다.  
생성자의 마지막 출력도 `Tanh`를 사용해 `[-1, 1]` 범위로 맞춥니다.


In [ ]:
# MNIST 이미지에 적용할 전처리 규칙을 정의합니다.
transform = transforms.Compose([
    # PIL 이미지를 PyTorch 텐서로 변환합니다.
    transforms.ToTensor(),

    # 이미지 픽셀 범위를 [0, 1]에서 [-1, 1]로 변환합니다.
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

# MNIST 학습 데이터셋을 생성합니다.
train_dataset = MNIST(
    root="./mnist",        # 데이터셋 저장 위치입니다.
    train=True,           # 학습용 데이터를 사용합니다.
    download=True,        # 데이터가 없으면 자동으로 다운로드합니다.
    transform=transform   # 위에서 정의한 전처리를 적용합니다.
)

# 학습용 DataLoader를 생성합니다.
train_loader = DataLoader(
    dataset=train_dataset,  # 사용할 데이터셋입니다.
    batch_size=BATCH_SIZE,  # 한 배치에 포함할 이미지 개수입니다.
    shuffle=True,           # 매 에포크마다 데이터를 섞습니다.
    num_workers=0,          # Windows와 Colab에서 안전하게 동작하도록 0으로 둡니다.
    pin_memory=torch.cuda.is_available() # GPU 사용 시 데이터 전송 효율을 높입니다.
)

# 학습 데이터 개수를 출력합니다.
print("학습 데이터 개수:", len(train_dataset))

# 첫 번째 배치를 가져옵니다.
real_images, _ = next(iter(train_loader))

# 첫 번째 배치의 이미지 모양을 출력합니다.
print("배치 이미지 모양:", real_images.shape)

In [ ]:
# 정규화된 이미지를 화면에 보기 위해 [0, 1] 범위로 되돌리는 함수입니다.
def show_image_grid(images, title="Image Grid"):
    # 이미지 텐서를 CPU로 이동합니다.
    images = images.detach().cpu()

    # 여러 이미지를 하나의 격자 이미지로 합칩니다.
    grid = vutils.make_grid(images[:25], nrow=5, normalize=True)

    # PyTorch 이미지 형식 [채널, 높이, 너비]를 matplotlib 형식 [높이, 너비, 채널]로 바꿉니다.
    np_grid = grid.permute(1, 2, 0).numpy()

    # 그림 크기를 지정합니다.
    plt.figure(figsize=(6, 6))

    # 이미지를 출력합니다.
    plt.imshow(np_grid)

    # 제목을 표시합니다.
    plt.title(title)

    # 축 눈금을 숨깁니다.
    plt.axis("off")

    # 이미지를 화면에 표시합니다.
    plt.show()

# 실제 MNIST 이미지 일부를 출력합니다.
show_image_grid(real_images, title="Real MNIST Images")

## 3. 생성자 모델 설계

생성자는 무작위 노이즈 벡터를 입력받아 28×28 흑백 이미지를 만듭니다.

입력 모양은 다음과 같습니다.

`[배치크기, 100, 1, 1]`

출력 모양은 다음과 같습니다.

`[배치크기, 1, 28, 28]`


In [ ]:
# DC-GAN 생성자 모델을 정의합니다.
class Generator(nn.Module):
    # 생성자에서 사용할 계층을 초기화합니다.
    def __init__(self, noise_dim=100, image_channels=1):
        # 부모 클래스 nn.Module의 초기화 메서드를 실행합니다.
        super().__init__()

        # 생성자는 ConvTranspose2d를 사용해 작은 노이즈를 큰 이미지로 확대합니다.
        self.net = nn.Sequential(
            # 입력 노이즈 [B, 100, 1, 1]을 [B, 256, 7, 7] 특징맵으로 확장합니다.
            nn.ConvTranspose2d(
                in_channels=noise_dim,
                out_channels=256,
                kernel_size=7,
                stride=1,
                padding=0,
                bias=False
            ),

            # 256개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(256),

            # 음수는 0으로 만들고 양수는 통과시켜 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # [B, 256, 7, 7]을 [B, 128, 14, 14]로 확대합니다.
            nn.ConvTranspose2d(
                in_channels=256,
                out_channels=128,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),

            # 128개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(128),

            # 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # [B, 128, 14, 14]를 [B, 64, 28, 28]로 확대합니다.
            nn.ConvTranspose2d(
                in_channels=128,
                out_channels=64,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),

            # 64개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(64),

            # 비선형성을 추가합니다.
            nn.ReLU(inplace=True),

            # [B, 64, 28, 28]을 [B, 1, 28, 28] 이미지로 변환합니다.
            nn.Conv2d(
                in_channels=64,
                out_channels=image_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),

            # 출력 픽셀 범위를 [-1, 1]로 맞춥니다.
            nn.Tanh()
        )

    # 생성자의 순전파 과정을 정의합니다.
    def forward(self, z):
        # 노이즈 z를 생성자 네트워크에 통과시켜 가짜 이미지를 생성합니다.
        return self.net(z)

## 4. 감별자 모델 설계

감별자는 이미지가 진짜인지 가짜인지 판별합니다.

입력 모양은 다음과 같습니다.

`[배치크기, 1, 28, 28]`

출력 모양은 다음과 같습니다.

`[배치크기]`

출력값은 확률이 아니라 `logit`입니다.  
따라서 마지막에 `Sigmoid`를 넣지 않습니다.


In [ ]:
# DC-GAN 감별자 모델을 정의합니다.
class Discriminator(nn.Module):
    # 감별자에서 사용할 계층을 초기화합니다.
    def __init__(self, image_channels=1):
        # 부모 클래스 nn.Module의 초기화 메서드를 실행합니다.
        super().__init__()

        # 감별자는 Conv2d를 사용해 이미지를 점점 작은 특징맵으로 압축합니다.
        self.net = nn.Sequential(
            # 입력 이미지 [B, 1, 28, 28]을 [B, 64, 14, 14] 특징맵으로 변환합니다.
            nn.Conv2d(
                in_channels=image_channels,
                out_channels=64,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),

            # GAN 감별자에서는 ReLU보다 LeakyReLU를 자주 사용합니다.
            nn.LeakyReLU(negative_slope=0.2, inplace=True),

            # [B, 64, 14, 14]를 [B, 128, 7, 7]로 압축합니다.
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),

            # 128개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(128),

            # 음수도 일부 통과시켜 기울기 소실을 줄입니다.
            nn.LeakyReLU(negative_slope=0.2, inplace=True),

            # [B, 128, 7, 7]을 [B, 256, 4, 4]로 변환합니다.
            nn.Conv2d(
                in_channels=128,
                out_channels=256,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),

            # 256개 특징맵의 분포를 안정화합니다.
            nn.BatchNorm2d(256),

            # 비선형성을 추가합니다.
            nn.LeakyReLU(negative_slope=0.2, inplace=True),

            # [B, 256, 4, 4]를 [B, 1, 1, 1]로 변환하여 최종 logit을 만듭니다.
            nn.Conv2d(
                in_channels=256,
                out_channels=1,
                kernel_size=4,
                stride=1,
                padding=0,
                bias=False
            )
        )

    # 감별자의 순전파 과정을 정의합니다.
    def forward(self, x):
        # 이미지를 감별자 네트워크에 통과시켜 logit을 계산합니다.
        out = self.net(x)

        # 출력 모양 [B, 1, 1, 1]을 [B]로 펼쳐 손실함수와 비교하기 쉽게 만듭니다.
        return out.view(-1)

In [ ]:
# DC-GAN 논문에서 자주 사용하는 가중치 초기화 함수를 정의합니다.
def weights_init(module):
    # 현재 계층의 클래스 이름을 문자열로 가져옵니다.
    classname = module.__class__.__name__

    # 합성곱 계층이면 정규분포로 가중치를 초기화합니다.
    if classname.find("Conv") != -1:
        # 평균 0, 표준편차 0.02의 정규분포로 가중치를 초기화합니다.
        nn.init.normal_(module.weight.data, 0.0, 0.02)

    # 배치 정규화 계층이면 가중치와 편향을 초기화합니다.
    elif classname.find("BatchNorm") != -1:
        # 배치 정규화의 scale 파라미터를 평균 1, 표준편차 0.02로 초기화합니다.
        nn.init.normal_(module.weight.data, 1.0, 0.02)

        # 배치 정규화의 bias 파라미터를 0으로 초기화합니다.
        nn.init.constant_(module.bias.data, 0)

# 생성자 객체를 생성합니다.
generator = Generator(NOISE_DIM, IMAGE_CHANNELS).to(device)

# 감별자 객체를 생성합니다.
discriminator = Discriminator(IMAGE_CHANNELS).to(device)

# 생성자 가중치를 초기화합니다.
generator.apply(weights_init)

# 감별자 가중치를 초기화합니다.
discriminator.apply(weights_init)

# 생성자 구조를 출력합니다.
print(generator)

# 감별자 구조를 출력합니다.
print(discriminator)

# 더미 노이즈를 생성합니다.
dummy_noise = torch.randn(4, NOISE_DIM, 1, 1).to(device)

# 더미 이미지 생성을 테스트합니다.
with torch.no_grad():
    dummy_fake = generator(dummy_noise)
    dummy_score = discriminator(dummy_fake)

# 생성자 출력 모양을 출력합니다.
print("생성자 출력 모양:", dummy_fake.shape)

# 감별자 출력 모양을 출력합니다.
print("감별자 출력 모양:", dummy_score.shape)

## 5. 모델 설계 상세 설명

### 5.1 DC-GAN 구조

DC-GAN은 합성곱 신경망을 사용하는 GAN입니다.

GAN에는 두 개의 모델이 있습니다.

첫째, 생성자는 무작위 노이즈를 입력받아 가짜 이미지를 만듭니다.  
둘째, 감별자는 입력 이미지가 진짜 이미지인지 생성자가 만든 가짜 이미지인지 판별합니다.

두 모델은 서로 경쟁합니다.  
생성자는 감별자를 속이려고 학습하고, 감별자는 진짜와 가짜를 더 잘 구분하려고 학습합니다.

### 5.2 생성자 구조

생성자는 `ConvTranspose2d`를 사용합니다.  
`ConvTranspose2d`는 작은 특징맵을 점점 큰 이미지로 확대하는 계층입니다.

이 예제의 생성자 흐름은 다음과 같습니다.

`[B, 100, 1, 1] → [B, 256, 7, 7] → [B, 128, 14, 14] → [B, 64, 28, 28] → [B, 1, 28, 28]`

마지막에는 `Tanh`를 사용합니다.  
그 이유는 실제 MNIST 이미지도 `Normalize(0.5, 0.5)`를 통해 `[-1, 1]` 범위로 정규화했기 때문입니다.

### 5.3 감별자 구조

감별자는 `Conv2d`를 사용합니다.  
입력 이미지를 점점 작은 특징맵으로 줄이면서 진짜/가짜를 판단하는 특징을 추출합니다.

이 예제의 감별자 흐름은 다음과 같습니다.

`[B, 1, 28, 28] → [B, 64, 14, 14] → [B, 128, 7, 7] → [B, 256, 4, 4] → [B]`

마지막 출력은 확률이 아니라 `logit`입니다.  
따라서 감별자 마지막에는 `Sigmoid`를 직접 넣지 않습니다.


## 6. 손실함수와 최적화 알고리즘 상세 설명

### 6.1 손실함수: BCEWithLogitsLoss

GAN의 감별자는 입력 이미지가 진짜인지 가짜인지 판별하는 이진 분류 문제를 풉니다.  
따라서 이진 분류 손실함수인 `BCEWithLogitsLoss`를 사용합니다.

`BCEWithLogitsLoss`는 내부적으로 다음 두 과정을 함께 처리합니다.

1. 감별자의 logit 출력에 Sigmoid 적용
2. Binary Cross Entropy 손실 계산

그래서 감별자 마지막 계층에 `Sigmoid`를 직접 넣지 않습니다.  
이 방식이 수치적으로 더 안정적입니다.

### 6.2 감별자 손실

감별자는 두 가지를 잘해야 합니다.

1. 진짜 이미지를 진짜라고 판단해야 합니다.
2. 가짜 이미지를 가짜라고 판단해야 합니다.

따라서 감별자 손실은 다음 두 손실의 합입니다.

`진짜 이미지 손실 + 가짜 이미지 손실`

### 6.3 생성자 손실

생성자는 감별자를 속이는 것이 목표입니다.  
즉, 생성자가 만든 가짜 이미지를 감별자가 진짜라고 판단하도록 학습합니다.

따라서 생성자 손실은 다음 기준으로 계산합니다.

`감별자가 가짜 이미지를 진짜라고 판단하게 만들기`

### 6.4 최적화 알고리즘: Adam

이 노트북에서는 DC-GAN에서 널리 사용하는 Adam 최적화 알고리즘을 사용합니다.

설정값은 다음과 같습니다.

- 학습률: `0.0002`
- beta1: `0.5`
- beta2: `0.999`

GAN은 학습이 불안정할 수 있으므로 일반적인 분류 모델보다 학습률을 작게 사용하는 것이 좋습니다.


In [ ]:
# BCEWithLogitsLoss는 감별자의 logit 출력과 정답 라벨을 비교합니다.
criterion = nn.BCEWithLogitsLoss()

# 생성자 전용 Adam 최적화 알고리즘을 생성합니다.
g_optimizer = optim.Adam(
    generator.parameters(),      # 생성자의 파라미터를 업데이트 대상으로 지정합니다.
    lr=LEARNING_RATE,            # 생성자 학습률을 지정합니다.
    betas=(BETA1, 0.999)         # DC-GAN에서 자주 사용하는 Adam beta 값을 지정합니다.
)

# 감별자 전용 Adam 최적화 알고리즘을 생성합니다.
d_optimizer = optim.Adam(
    discriminator.parameters(),  # 감별자의 파라미터를 업데이트 대상으로 지정합니다.
    lr=LEARNING_RATE,            # 감별자 학습률을 지정합니다.
    betas=(BETA1, 0.999)         # DC-GAN에서 자주 사용하는 Adam beta 값을 지정합니다.
)

# 생성 이미지 변화를 비교하기 위해 고정 노이즈를 생성합니다.
fixed_noise = torch.randn(25, NOISE_DIM, 1, 1, device=device)

In [ ]:
# 생성 이미지를 저장하고 출력하는 함수를 정의합니다.
def save_generated_images(epoch, generator, fixed_noise, output_dir):
    # 생성자를 평가 모드로 전환합니다.
    generator.eval()

    # 이미지 생성에는 기울기 계산이 필요 없으므로 no_grad를 사용합니다.
    with torch.no_grad():
        # 고정 노이즈로 가짜 이미지를 생성합니다.
        fake_images = generator(fixed_noise).detach().cpu()

    # 생성된 이미지를 PNG 파일로 저장할 경로를 만듭니다.
    save_path = os.path.join(output_dir, f"epoch_{epoch:03d}.png")

    # 생성된 이미지를 5x5 그리드로 저장합니다.
    vutils.save_image(
        fake_images,       # 저장할 이미지 텐서입니다.
        save_path,         # 저장 경로입니다.
        nrow=5,            # 한 줄에 5개 이미지를 배치합니다.
        normalize=True     # [-1, 1] 이미지를 보기 좋게 [0, 1]로 변환합니다.
    )

    # 저장된 이미지를 화면에도 출력합니다.
    show_image_grid(fake_images, title=f"Generated Images - Epoch {epoch}")

    # 저장 경로를 반환합니다.
    return save_path

In [ ]:
# GAN을 학습하는 함수를 정의합니다.
def train_gan(generator, discriminator, train_loader, criterion, g_optimizer, d_optimizer, epochs, device):
    # 학습 시작 시간을 기록합니다.
    start_time = time()

    # 손실 기록을 저장할 리스트를 생성합니다.
    g_losses = []
    d_losses = []

    # 지정한 에포크 수만큼 반복합니다.
    for epoch in range(1, epochs + 1):
        # 생성자를 학습 모드로 전환합니다.
        generator.train()

        # 감별자를 학습 모드로 전환합니다.
        discriminator.train()

        # 에포크 단위 생성자 손실 합계를 초기화합니다.
        epoch_g_loss = 0.0

        # 에포크 단위 감별자 손실 합계를 초기화합니다.
        epoch_d_loss = 0.0

        # 배치 개수를 초기화합니다.
        batch_count = 0

        # DataLoader에서 실제 이미지를 미니배치 단위로 가져옵니다.
        for batch_idx, (real_images, _) in enumerate(train_loader):
            # 실제 이미지를 학습 장치로 이동합니다.
            real_images = real_images.to(device)

            # 현재 배치 크기를 가져옵니다.
            current_batch_size = real_images.size(0)

            # 진짜 이미지의 정답 라벨을 1로 만듭니다.
            real_labels = torch.ones(current_batch_size, device=device)

            # 가짜 이미지의 정답 라벨을 0으로 만듭니다.
            fake_labels = torch.zeros(current_batch_size, device=device)

            # =========================
            # 1단계: 감별자 학습
            # =========================

            # 감별자 기울기를 초기화합니다.
            d_optimizer.zero_grad(set_to_none=True)

            # 감별자가 진짜 이미지를 보고 출력한 logit을 계산합니다.
            real_logits = discriminator(real_images)

            # 진짜 이미지를 진짜로 맞히는 손실을 계산합니다.
            d_real_loss = criterion(real_logits, real_labels)

            # 생성자 입력용 무작위 노이즈를 생성합니다.
            noise = torch.randn(current_batch_size, NOISE_DIM, 1, 1, device=device)

            # 생성자가 가짜 이미지를 만듭니다.
            fake_images = generator(noise)

            # 감별자가 가짜 이미지를 보고 출력한 logit을 계산합니다.
            # detach()를 사용하여 감별자 학습 중 생성자 가중치는 업데이트되지 않게 합니다.
            fake_logits = discriminator(fake_images.detach())

            # 가짜 이미지를 가짜로 맞히는 손실을 계산합니다.
            d_fake_loss = criterion(fake_logits, fake_labels)

            # 감별자의 전체 손실은 진짜 손실과 가짜 손실의 합입니다.
            d_loss = d_real_loss + d_fake_loss

            # 감별자 손실을 기준으로 역전파를 수행합니다.
            d_loss.backward()

            # 감별자 파라미터를 업데이트합니다.
            d_optimizer.step()

            # =========================
            # 2단계: 생성자 학습
            # =========================

            # 생성자 기울기를 초기화합니다.
            g_optimizer.zero_grad(set_to_none=True)

            # 생성자가 만든 가짜 이미지를 감별자에 다시 넣습니다.
            gen_logits = discriminator(fake_images)

            # 생성자는 가짜 이미지를 진짜로 판별받는 것이 목표이므로 라벨을 1로 둡니다.
            g_loss = criterion(gen_logits, real_labels)

            # 생성자 손실을 기준으로 역전파를 수행합니다.
            g_loss.backward()

            # 생성자 파라미터를 업데이트합니다.
            g_optimizer.step()

            # 현재 배치의 생성자 손실을 누적합니다.
            epoch_g_loss += g_loss.item()

            # 현재 배치의 감별자 손실을 누적합니다.
            epoch_d_loss += d_loss.item()

            # 배치 개수를 증가시킵니다.
            batch_count += 1

            # 일정 배치마다 학습 상태를 출력합니다.
            if (batch_idx + 1) % 100 == 0:
                print(
                    f"Epoch [{epoch}/{epochs}] "
                    f"Batch [{batch_idx + 1}/{len(train_loader)}] "
                    f"D Loss: {d_loss.item():.4f} "
                    f"G Loss: {g_loss.item():.4f}"
                )

        # 에포크 평균 생성자 손실을 계산합니다.
        avg_g_loss = epoch_g_loss / batch_count

        # 에포크 평균 감별자 손실을 계산합니다.
        avg_d_loss = epoch_d_loss / batch_count

        # 평균 생성자 손실을 기록합니다.
        g_losses.append(avg_g_loss)

        # 평균 감별자 손실을 기록합니다.
        d_losses.append(avg_d_loss)

        # 에포크 결과를 출력합니다.
        print(f"\nEpoch {epoch}/{epochs} 완료 | D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f}")

        # 현재 에포크의 생성 이미지를 저장하고 출력합니다.
        save_path = save_generated_images(epoch, generator, fixed_noise, OUTPUT_DIR)

        # 저장 경로를 출력합니다.
        print("생성 이미지 저장:", save_path)

    # 학습 종료 시간을 기록합니다.
    end_time = time()

    # 전체 학습 시간을 출력합니다.
    print(f"\n총 학습 시간: {end_time - start_time:.1f}초")

    # 손실 기록을 반환합니다.
    return g_losses, d_losses

## 7. GAN 학습 실행

GAN 학습은 일반 분류 모델보다 불안정할 수 있습니다.  
손실값이 단순히 계속 감소하지 않을 수 있으며, 생성 이미지의 품질을 함께 확인하는 것이 중요합니다.


In [ ]:
# GAN 학습을 실행합니다.
g_losses, d_losses = train_gan(
    generator=generator,             # 학습할 생성자입니다.
    discriminator=discriminator,     # 학습할 감별자입니다.
    train_loader=train_loader,       # MNIST 학습 데이터 로더입니다.
    criterion=criterion,             # 손실함수입니다.
    g_optimizer=g_optimizer,         # 생성자 최적화 알고리즘입니다.
    d_optimizer=d_optimizer,         # 감별자 최적화 알고리즘입니다.
    epochs=EPOCHS,                   # 학습 에포크 수입니다.
    device=device                    # 학습 장치입니다.
)

## 8. 손실 그래프 확인

GAN에서는 생성자 손실과 감별자 손실이 서로 경쟁적으로 변합니다.  
두 손실 중 하나만 계속 낮아지는 것이 항상 좋은 것은 아닙니다.

생성 이미지가 점점 숫자처럼 보이는지도 함께 확인해야 합니다.


In [ ]:
# 에포크 번호를 생성합니다.
epochs_range = range(1, EPOCHS + 1)

# 손실 그래프를 그릴 그림을 생성합니다.
plt.figure(figsize=(8, 5))

# 생성자 손실 그래프를 그립니다.
plt.plot(epochs_range, g_losses, marker="o", label="Generator Loss")

# 감별자 손실 그래프를 그립니다.
plt.plot(epochs_range, d_losses, marker="o", label="Discriminator Loss")

# 그래프 제목을 지정합니다.
plt.title("DC-GAN Loss Curve")

# x축 이름을 지정합니다.
plt.xlabel("Epoch")

# y축 이름을 지정합니다.
plt.ylabel("Loss")

# 범례를 표시합니다.
plt.legend()

# 격자를 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()

## 9. 새 이미지 생성

학습된 생성자에 새로운 무작위 노이즈를 넣어 숫자 이미지를 생성합니다.


In [ ]:
# 생성할 이미지 개수를 지정합니다.
num_generate = 25

# 새로운 무작위 노이즈를 생성합니다.
new_noise = torch.randn(num_generate, NOISE_DIM, 1, 1, device=device)

# 생성자를 평가 모드로 전환합니다.
generator.eval()

# 이미지 생성에는 기울기 계산이 필요 없으므로 no_grad를 사용합니다.
with torch.no_grad():
    # 새로운 노이즈로 가짜 이미지를 생성합니다.
    generated_images = generator(new_noise)

# 생성된 이미지를 화면에 출력합니다.
show_image_grid(generated_images, title="New Generated Images")

## 10. 모델 저장과 불러오기

GAN은 생성자와 감별자 모델을 각각 저장할 수 있습니다.  
실제로 이미지를 생성할 때는 주로 생성자 모델을 사용합니다.


In [ ]:
# 생성자 가중치를 저장할 파일 이름입니다.
GENERATOR_PATH = "dcgan_generator.pth"

# 감별자 가중치를 저장할 파일 이름입니다.
DISCRIMINATOR_PATH = "dcgan_discriminator.pth"

# 생성자 가중치를 저장합니다.
torch.save(generator.state_dict(), GENERATOR_PATH)

# 감별자 가중치를 저장합니다.
torch.save(discriminator.state_dict(), DISCRIMINATOR_PATH)

# 저장 완료 메시지를 출력합니다.
print("생성자 저장 완료:", GENERATOR_PATH)

# 저장 완료 메시지를 출력합니다.
print("감별자 저장 완료:", DISCRIMINATOR_PATH)

# 같은 구조의 새 생성자 모델을 생성합니다.
loaded_generator = Generator(NOISE_DIM, IMAGE_CHANNELS).to(device)

# 저장된 생성자 가중치를 불러옵니다.
loaded_generator.load_state_dict(torch.load(GENERATOR_PATH, map_location=device))

# 불러온 생성자를 평가 모드로 전환합니다.
loaded_generator.eval()

# 불러오기 완료 메시지를 출력합니다.
print("생성자 모델 불러오기 완료")

## 11. 성능 개선 실험 방향

생성 이미지 품질을 높이려면 다음을 실험할 수 있습니다.

1. `EPOCHS`를 20 이상으로 늘립니다.
2. `BATCH_SIZE`를 64 또는 128로 조정합니다.
3. 생성자와 감별자의 채널 수를 늘립니다.
4. 학습률을 `0.0001` 또는 `0.0002` 범위에서 조정합니다.
5. 생성자와 감별자의 학습 균형을 확인합니다.
6. 감별자가 너무 강하면 생성자가 학습하지 못할 수 있습니다.
7. 생성자가 너무 강하면 감별자가 진짜/가짜를 구분하지 못할 수 있습니다.
